### Chunking

In [1]:
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [3]:
md_path = Path("../output_cleaned_final.md")
markdown_text = md_path.read_text(encoding="utf-8")

In [4]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_text)
len(md_header_splits)


Header-based splits created: 49


In [13]:
for split in md_header_splits[:5]:
    print("Content :\n", split.page_content)
    print("Metadata:", split.metadata)
    print("\n\n")

Content :
 **Machine learning** ( **ML** ) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions.  Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine. The application of ML to business problems is known as predictive analytics.
Statistics and mathematical optimisation (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.
From a theoretical viewpoint, probably approxim

In [14]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

final_splits = recursive_splitter.split_documents(md_header_splits)
len(final_splits)


73

In [15]:
for i, chunk in enumerate(final_splits):  
    print(f"\nChunk {i}")
    print(chunk.page_content)
    print("Metadata:", chunk.metadata)



Chunk 0
**Machine learning** ( **ML** ) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions.  Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine. The application of ML to business problems is known as predictive analytics.
Statistics and mathematical optimisation (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.
From a theoretical viewpoint, probably approximat

### Embedding

In [22]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [23]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=client.api_key
)

vectorstore = FAISS.from_documents(
    documents=final_splits,
    embedding=embeddings
)
vectorstore.save_local("../ml_vector_db")
